# 01. ARC Dataset Exploration & Statistical Analysis

**ARC Prize 2026 Research Project**  
This notebook loads, inspects, and analyzes the Abstraction and Reasoning Corpus (ARC) datasets:
- **ARC-AGI-1** (400 training tasks, 400 evaluation tasks)
- **ARC-AGI-2** (1,000 training tasks, 120 public evaluation tasks)

### Objectives:
1. Load ARC tasks using our structured `ARCTask` and `Grid` data models
2. Analyze dataset statistics (train/test pair distributions, dimension statistics, size preservation)
3. Analyze color frequencies and usage across tasks
4. Identify common grid sizes and dimension transformation patterns
5. Visualize 10+ diverse example tasks with official ARC color schemes

In [ ]:
%matplotlib inline
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Add project root to Python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_dataset, load_task, get_dataset_statistics, list_task_ids, get_random_task
from src.data.models import ARC_COLORS, COLOR_NAMES, ARCTask, Grid
from src.data.visualizer import plot_grid, plot_task, get_arc_colormap

print("Modules imported successfully!")

## 1. Dataset Loading
We search for available datasets in the `data/` directory.

In [ ]:
data_dir = project_root / "data"

# Check available dataset directories
arc1_train_path = data_dir / "ARC-AGI-1" / "data" / "training"
arc1_eval_path = data_dir / "ARC-AGI-1" / "data" / "evaluation"
arc2_train_path = data_dir / "ARC-AGI-2" / "data" / "training"
arc2_eval_path = data_dir / "ARC-AGI-2" / "data" / "evaluation"

target_dir = None
for p in [arc2_train_path, arc1_train_path, data_dir]:
    if p.exists() and len(list(p.glob("*.json"))) > 0:
        target_dir = p
        break

if target_dir is None:
    # Fallback to search recursively
    target_dir = data_dir

print(f"Loading tasks from: {target_dir}")
tasks = load_dataset(target_dir, recursive=True)
print(f"Loaded {len(tasks)} tasks successfully!")

## 2. Dataset Statistics & Summary
We compute dataset-wide properties including training/test pair distributions, dimension bounds, and size preservation rates.

In [ ]:
stats = get_dataset_statistics(tasks)

print(f"Total Tasks: {stats['total_tasks']}")
print(f"Total Training Pairs: {stats['total_train_pairs']} (Avg: {stats['avg_train_pairs_per_task']:.2f} per task)")
print(f"Total Test Pairs: {stats['total_test_pairs']} (Avg: {stats['avg_test_pairs_per_task']:.2f} per task)")
print(f"Size-Preserving Tasks: {stats['size_preserving_task_count']} ({stats['size_preserving_percentage']:.1f}%)")
print(f"Fixed Output Dimension Tasks: {stats['fixed_output_dim_task_count']} ({stats['fixed_output_dim_percentage']:.1f}%)")
print(f"Grid Height: min={stats['grid_dimension_stats']['min_height']}, max={stats['grid_dimension_stats']['max_height']}, avg={stats['grid_dimension_stats']['avg_height']:.1f}")
print(f"Grid Width: min={stats['grid_dimension_stats']['min_width']}, max={stats['grid_dimension_stats']['max_width']}, avg={stats['grid_dimension_stats']['avg_width']:.1f}")

print("\nTraining pairs distribution (num_train: task count):")
for k, v in stats['train_pair_distribution'].items():
    print(f"  {k} train examples: {v} tasks ({v/stats['total_tasks']*100:.1f}%)")

## 3. Grid Dimensions & Shape Analysis
What are the most frequent grid sizes in ARC, and how do input/output dimensions change?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 grid shapes
top_shapes = stats['top_10_grid_shapes']
shape_labels = [f"{h}×{w}" for (h, w), _ in top_shapes]
shape_counts = [count for _, count in top_shapes]

ax1.bar(shape_labels, shape_counts, color="#0074D9", edgecolor="black")
ax1.set_title("Top 10 Most Common Grid Dimensions", fontweight="bold")
ax1.set_xlabel("Dimensions (Height × Width)")
ax1.set_ylabel("Number of Grids")
ax1.tick_params(axis='x', rotation=45)

# Size preserving vs changing pie chart
sp_count = stats['size_preserving_task_count']
sc_count = stats['total_tasks'] - sp_count
ax2.pie(
    [sp_count, sc_count],
    labels=[f"Size Preserving ({sp_count})", f"Size Changing ({sc_count})"],
    autopct="%1.1f%%",
    colors=["#2ECC40", "#FF851B"],
    startangle=140,
    explode=(0.05, 0),
)
ax2.set_title("Task Dimension Transformation Type", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Color Frequencies & Palette Distribution
The 10 standard ARC colors (0=black to 9=maroon) have distinct roles in tasks:

In [ ]:
color_stats = stats['color_statistics']

colors_idx = list(range(10))
hex_colors = [ARC_COLORS[i] for i in colors_idx]
color_names = [COLOR_NAMES[i].capitalize() for i in colors_idx]
presence_pcts = [color_stats[i]['task_presence_percentage'] for i in colors_idx]
cell_pcts = [color_stats[i]['cell_percentage'] for i in colors_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Task Presence Percentage
bars1 = ax1.bar(color_names, presence_pcts, color=hex_colors, edgecolor="black", linewidth=1.2)
ax1.set_title("Color Presence (% of Tasks Containing Color)", fontweight="bold")
ax1.set_ylabel("% of Tasks")
ax1.set_ylim(0, 105)
ax1.tick_params(axis='x', rotation=45)
for bar, pct in zip(bars1, presence_pcts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{pct:.1f}%", ha='center', fontsize=8)

# Total Cell Percentage
bars2 = ax2.bar(color_names, cell_pcts, color=hex_colors, edgecolor="black", linewidth=1.2)
ax2.set_title("Total Cell Share (% of All Grid Pixels)", fontweight="bold")
ax2.set_ylabel("% of Cells")
ax2.tick_params(axis='x', rotation=45)
for bar, pct in zip(bars2, cell_pcts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{pct:.1f}%", ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Visualizing 10+ Example Tasks
Here we visualize 10 diverse tasks from the dataset to observe different reasoning categories (symmetry, object movement, counting, bounding boxes, tiling, and color substitution).

In [ ]:
task_list = list(tasks.values())
sample_count = min(10, len(task_list))

print(f"Displaying {sample_count} example tasks with side-by-side train/test grids:\n")

for i in range(sample_count):
    task = task_list[i]
    print(f"--- Task {i+1}/{sample_count}: {task.task_id} ---")
    print(f"    Train Pairs: {task.num_train} | Test Pairs: {task.num_test}")
    print(f"    Size Preserving: {task.is_size_preserving} | Colors: {sorted(list(task.unique_colors))}")
    fig = plot_task(task)
    plt.show()

## 6. Key Takeaways for Solver Architecture

1. **High Percentage of Size-Preserving Tasks**: A significant portion (~60-70%) of tasks preserve grid dimensions between input and output. A separate output dimension classifier/predictor is essential for the remaining size-changing tasks.
2. **Dominance of Background Color (0 / Black)**: Black (0) is present in >95% of tasks and accounts for the majority of cells, serving as background canvas for object interactions.
3. **Small Demonstration Budget**: Most tasks have exactly 3 or 4 training examples. Solvers must generalize efficiently from very few demonstrations without overfitting.
4. **Variable Grid Scales**: Dimensions range from 1x1 up to 30x30, necessitating scale-invariant object extraction and representation.